# AgentCore Web Search Tool — Gateway Setup and Raw MCP Calls

## Overview

The **Amazon Bedrock AgentCore Web Search Tool** eliminates a core limitation of AI agents: knowledge frozen at training time. This notebook creates all the infrastructure needed and verifies it with direct MCP protocol calls — no agent framework yet, just raw tool discovery and invocation.

You will:
1. Create the IAM service role for the Gateway
2. Set up Amazon Cognito for inbound authentication
3. Create an AgentCore Gateway with MCP protocol
4. Add a Web Search Tool connector target
5. Verify with direct `tools/list` and `tools/call` MCP calls

The environment variables printed at the end are used by notebooks 02 and 03.

![Tutorial Architecture](../images/tutorial-architecture.png)

### Tutorial Details

| Information | Details |
|:------------|:--------|
| Tutorial type | Interactive (Jupyter Notebook) |
| AgentCore components | AgentCore Gateway |
| Gateway target type | Connector (`web-search`) |
| Inbound Auth IdP | Amazon Cognito (`client_credentials` flow) |
| Outbound Auth | Automatic (Gateway IAM Role) |
| Tutorial vertical | Genral Purpose - applicable in any industry |
| Example complexity | Easy |
| SDK used | boto3 |

### Authentication Architecture

![Inbound and Outbound Auth](../images/inbound-and-outbound-auth.png)

- **Inbound**: Cognito validates the OAuth token your agent passes to the Gateway
- **Outbound**: The Gateway IAM role authenticates to the Web Search backend automatically

## Prerequisites
- Jupyter notebook (Python kernel)
- Python 3.10+ or later
- AWS credentials  - choode one 
  - **Account ID + Role Name** — the notebook will call `sts:AssumeRole` to get temporary credentials (recommended for cross-account or Isengard access)
  - **Pre-configured credentials** — via AWS CLI profile, environment variables, or instance role
- AWS account with access to  Bedrock AgentCore Gateway, Cognito, and IAM
- Access to Bedrock inference models in us-east-1

> **Region**: The Web Search Tool connector is available in **us-east-1** only.

### Required IAM permissions

~~~json
{
  "Effect": "Allow",
  "Action": [
    "iam:CreateRole", "iam:PutRolePolicy", "iam:GetRole",
    "cognito-idp:CreateUserPool", "cognito-idp:CreateUserPoolDomain",
    "cognito-idp:CreateResourceServer", "cognito-idp:CreateUserPoolClient",
    "cognito-idp:ListUserPools", "cognito-idp:ListUserPoolClients",
    "cognito-idp:DescribeUserPoolClient", "cognito-idp:DescribeResourceServer",
    "bedrock-agentcore:CreateGateway", "bedrock-agentcore:GetGateway",
    "bedrock-agentcore:CreateGatewayTarget", "bedrock-agentcore:ListGatewayTargets"
  ],
  "Resource": "*"
}
~~~

## 1. Install Dependencies

In [ ]:
!pip install --upgrade -r requirements.txt --quiet

## 2. Configure AWS Region and Imports

In [ ]:
import boto3
import json
import os
import time
import requests

# Web Search connector is only available in us-east-1
REGION = "us-east-1"
os.environ["AWS_DEFAULT_REGION"] = REGION

# Verify credentials
sts_client = boto3.client("sts", region_name=REGION)
identity = sts_client.get_caller_identity()
ACCOUNT_ID = identity["Account"]
print(f"Account ID : {ACCOUNT_ID}")
print(f"Identity   : {identity['Arn']}")
print(f"Region     : {REGION}")

## 3. Create the Gateway Service Role

The Gateway needs an IAM role allowing the AgentCore service to perform actions on your behalf. Two permissions are required:
- `bedrock-agentcore:InvokeGateway` — invoke the Gateway itself
- `bedrock-agentcore:InvokeWebSearch` — authorize web search calls to `arn:aws:bedrock-agentcore:<region>:aws:tool/web-search.v1`

In [ ]:
GATEWAY_NAME = "web-search-gateway"
GATEWAY_ROLE_NAME = f"agentcore-{GATEWAY_NAME}-role"

iam_client = boto3.client("iam")

assume_role_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
        "Action": "sts:AssumeRole",
        "Condition": {
            "StringEquals": {"aws:SourceAccount": ACCOUNT_ID},
            "ArnLike": {"aws:SourceArn": f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:*"}
        }
    }]
}

try:
    role_resp = iam_client.create_role(
        RoleName=GATEWAY_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(assume_role_policy)
    )
    print(f"Created role: {GATEWAY_ROLE_NAME}")
    time.sleep(10)  # Allow IAM to propagate
except iam_client.exceptions.EntityAlreadyExistsException:
    role_resp = iam_client.get_role(RoleName=GATEWAY_ROLE_NAME)
    print(f"Role already exists: {GATEWAY_ROLE_NAME}")

GATEWAY_ROLE_ARN = role_resp["Role"]["Arn"]

iam_client.put_role_policy(
    RoleName=GATEWAY_ROLE_NAME,
    PolicyName="WebSearchGatewayPolicy",
    PolicyDocument=json.dumps({
        "Version": "2012-10-17",
        "Statement": [
            {
                "Sid": "InvokeGateway",
                "Effect": "Allow",
                "Action": "bedrock-agentcore:InvokeGateway",
                "Resource": f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:gateway/*"
            },
            {
                "Sid": "InvokeWebSearch",
                "Effect": "Allow",
                "Action": "bedrock-agentcore:InvokeWebSearch",
                "Resource": f"arn:aws:bedrock-agentcore:{REGION}:aws:tool/web-search.v1"
            }
        ]
    })
)
print(f"Gateway Role ARN: {GATEWAY_ROLE_ARN}")
print("Permissions attached ✓")

## 4. Configure Inbound Authentication with Amazon Cognito

We create a Cognito User Pool with a machine-to-machine (M2M) client using the `client_credentials` OAuth flow. The Gateway will validate JWT tokens issued by this pool.

In [ ]:
cognito = boto3.client("cognito-idp", region_name=REGION)

USER_POOL_NAME = "agentcore-websearch-pool"
RESOURCE_SERVER_ID = "agentcore-websearch"
SCOPES = [{"ScopeName": "invoke", "ScopeDescription": "Invoke gateway"}]
SCOPE_NAMES = [f"{RESOURCE_SERVER_ID}/{s['ScopeName']}" for s in SCOPES]
SCOPE_STRING = " ".join(SCOPE_NAMES)

# Find or create user pool
USER_POOL_ID = None
for pool in cognito.list_user_pools(MaxResults=60)["UserPools"]:
    if pool["Name"] == USER_POOL_NAME:
        USER_POOL_ID = pool["Id"]
        break

if USER_POOL_ID is None:
    resp = cognito.create_user_pool(PoolName=USER_POOL_NAME)
    USER_POOL_ID = resp["UserPool"]["Id"]
    domain = USER_POOL_ID.replace("_", "").lower()
    cognito.create_user_pool_domain(Domain=domain, UserPoolId=USER_POOL_ID)
    print(f"Created user pool: {USER_POOL_ID}")
else:
    print(f"User pool exists: {USER_POOL_ID}")

# Create resource server
try:
    cognito.describe_resource_server(UserPoolId=USER_POOL_ID, Identifier=RESOURCE_SERVER_ID)
except cognito.exceptions.ResourceNotFoundException:
    cognito.create_resource_server(
        UserPoolId=USER_POOL_ID, Identifier=RESOURCE_SERVER_ID,
        Name="WebSearch Gateway Resource Server", Scopes=SCOPES
    )
print("Resource server ensured ✓")

# Find or create M2M client
CLIENT_ID, CLIENT_SECRET = None, None
for c in cognito.list_user_pool_clients(UserPoolId=USER_POOL_ID, MaxResults=60)["UserPoolClients"]:
    if c["ClientName"] == "agentcore-websearch-client":
        desc = cognito.describe_user_pool_client(UserPoolId=USER_POOL_ID, ClientId=c["ClientId"])
        CLIENT_ID = c["ClientId"]
        CLIENT_SECRET = desc["UserPoolClient"]["ClientSecret"]
        break

if CLIENT_ID is None:
    created = cognito.create_user_pool_client(
        UserPoolId=USER_POOL_ID, ClientName="agentcore-websearch-client",
        GenerateSecret=True, AllowedOAuthFlows=["client_credentials"],
        AllowedOAuthScopes=SCOPE_NAMES, AllowedOAuthFlowsUserPoolClient=True,
        SupportedIdentityProviders=["COGNITO"],
        ExplicitAuthFlows=["ALLOW_REFRESH_TOKEN_AUTH"]
    )
    CLIENT_ID = created["UserPoolClient"]["ClientId"]
    CLIENT_SECRET = created["UserPoolClient"]["ClientSecret"]
    print(f"Created client: {CLIENT_ID}")
else:
    print(f"Client exists: {CLIENT_ID}")

COGNITO_DOMAIN = USER_POOL_ID.replace("_", "").lower()
DISCOVERY_URL = f"https://cognito-idp.{REGION}.amazonaws.com/{USER_POOL_ID}/.well-known/openid-configuration"
print(f"Discovery URL: {DISCOVERY_URL}")

## 5. Create the AgentCore Gateway

In [ ]:
gateway_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

create_resp = gateway_client.create_gateway(
    name=GATEWAY_NAME,
    roleArn=GATEWAY_ROLE_ARN,
    protocolType="MCP",
    protocolConfiguration={
        "mcp": {"supportedVersions": ["2025-03-26"], "searchType": "SEMANTIC"}
    },
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration={
        "customJWTAuthorizer": {
            "allowedClients": [CLIENT_ID],
            "discoveryUrl": DISCOVERY_URL
        }
    },
    description="AgentCore Gateway with Web Search Tool"
)

GATEWAY_ID = create_resp["gatewayId"]
GATEWAY_URL = create_resp["gatewayUrl"]
print(f"Gateway ID  : {GATEWAY_ID}")
print(f"Gateway URL : {GATEWAY_URL}")

# Wait for READY
for _ in range(30):
    status = gateway_client.get_gateway(gatewayIdentifier=GATEWAY_ID)["status"]
    if status == "READY":
        break
    time.sleep(5)
print(f"Gateway status: {status}")

## 6. Add the Web Search Connector Target

The Web Search Tool uses the `connector` target type. Specify `connectorId: "web-search"` and the Gateway handles schema management, endpoint resolution, and outbound auth automatically.

In [ ]:
target_resp = gateway_client.create_gateway_target(
    name="web-search-tool",
    gatewayIdentifier=GATEWAY_ID,
    targetConfiguration={
        "mcp": {
            "connector": {
                "source": {"connectorId": "web-search"},
                "configurations": [{"name": "WebSearch", "parameterValues": {}}]
            }
        }
    },
    credentialProviderConfigurations=[
        {"credentialProviderType": "GATEWAY_IAM_ROLE"}
    ]
)

TARGET_ID = target_resp["targetId"]
print(f"Target ID: {TARGET_ID}")

# Wait for target READY
for _ in range(30):
    targets = gateway_client.list_gateway_targets(gatewayIdentifier=GATEWAY_ID)
    statuses = [t["status"] for t in targets["items"]]
    print(f"  Target statuses: {statuses}")
    if all(s == "READY" for s in statuses):
        print("\n✅ All targets READY")
        break
    time.sleep(5)

## 7. Get an OAuth Token

Retrieve a Cognito access token using the `client_credentials` flow. This token is passed as a Bearer header on every MCP call.

In [ ]:
def get_token():
    """Retrieve a fresh OAuth access token from Cognito."""
    url = f"https://{COGNITO_DOMAIN}.auth.{REGION}.amazoncognito.com/oauth2/token"
    resp = requests.post(
        url,
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        data={
            "grant_type": "client_credentials",
            "client_id": CLIENT_ID,
            "client_secret": CLIENT_SECRET,
            "scope": SCOPE_STRING
        }
    )
    resp.raise_for_status()
    return resp.json()["access_token"]

# Allow time for Cognito domain propagation
print("Waiting for Cognito domain propagation...")
time.sleep(15)

token = get_token()
print(f"Token obtained ✓ (first 20 chars: {token[:20]}...)")

## 8. Verify: Raw MCP Tool Discovery (`tools/list`)

Call the Gateway directly using MCP — no agent framework. This confirms the WebSearch tool is available and shows its input schema.

In [ ]:
from mcp.client.streamable_http import streamablehttp_client
from strands.tools.mcp.mcp_client import MCPClient


def create_transport():
    return streamablehttp_client(
        GATEWAY_URL,
        headers={"Authorization": f"Bearer {token}"}
    )


mcp_client = MCPClient(create_transport)

with mcp_client:
    tools = mcp_client.list_tools_sync()
    print(f"Discovered {len(tools)} tool(s):\n")
    for tool in tools:
        spec = tool.tool_spec
        print(f"  Name        : {spec['name']}")
        print(f"  Description : {spec.get('description', 'N/A')}")
        schema = json.dumps(spec.get('inputSchema', {}), indent=4)
        print(f"  Input schema: {schema[:300]}")
        print()

## 9. Verify: Raw MCP Tool Invocation (`tools/call`)

Invoke the WebSearch tool directly and inspect the structured response.

In [ ]:
TEST_QUERY = "What is today's news around the world?"

with mcp_client:
    tools = mcp_client.list_tools_sync()
    ws_tool_name = next(t.tool_name for t in tools if "WebSearch" in t.tool_name)

    print(f"Invoking: {ws_tool_name}")
    print(f"Query   : {TEST_QUERY}\n")

    result = mcp_client.call_tool_sync("verify-call", ws_tool_name, {"query": TEST_QUERY})

    for content in result.get("content", [result] if not isinstance(result, list) else result):
        if isinstance(content, dict) and "text" in content:
            try:
                parsed = json.loads(content["text"])
                print(json.dumps(parsed, indent=2))
            except (ValueError, TypeError):
                print(content["text"][:2000])
        else:
            print(content)

## 10. Save Environment Variables for Notebooks 02 and 03

Export these before running the next notebooks.

In [ ]:
print("# Copy and export these environment variables:\n")
print(f'export AGENTCORE_GATEWAY_URL="{GATEWAY_URL}"')
print(f'export COGNITO_DOMAIN="{COGNITO_DOMAIN}"')
print(f'export COGNITO_CLIENT_ID="{CLIENT_ID}"')
print(f'export COGNITO_CLIENT_SECRET="{CLIENT_SECRET}"')
print(f'export COGNITO_SCOPE="{SCOPE_STRING}"')
print(f'export AWS_DEFAULT_REGION="{REGION}"')
print(f'\n# For cleanup:')
print(f'# Gateway ID   : {GATEWAY_ID}')
print(f'# User Pool ID : {USER_POOL_ID}')
print(f'# IAM Role     : {GATEWAY_ROLE_NAME}')

## Cleanup (Optional)

Run this cell only when you are done with **all** tutorials in this workshop. The Gateway and Cognito pool are reused by notebooks 02 and 03.

In [ ]:
# Uncomment to delete all resources

# # Delete Gateway targets and Gateway
# for item in gateway_client.list_gateway_targets(gatewayIdentifier=GATEWAY_ID, maxResults=100)["items"]:
#     gateway_client.delete_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=item["targetId"])
#     print(f"Deleted target: {item['name']}")
# time.sleep(10)
# gateway_client.delete_gateway(gatewayIdentifier=GATEWAY_ID)
# print(f"Deleted gateway: {GATEWAY_ID}")

# # Delete Cognito
# cognito.delete_user_pool_domain(Domain=COGNITO_DOMAIN, UserPoolId=USER_POOL_ID)
# cognito.delete_user_pool(UserPoolId=USER_POOL_ID)
# print(f"Deleted user pool: {USER_POOL_ID}")

# # Delete IAM role
# iam_client.delete_role_policy(RoleName=GATEWAY_ROLE_NAME, PolicyName="WebSearchGatewayPolicy")
# iam_client.delete_role(RoleName=GATEWAY_ROLE_NAME)
# print(f"Deleted role: {GATEWAY_ROLE_NAME}")

print("Uncomment the lines above to delete resources.")

## Conclusion

In this notebook you:
- Created an AgentCore Gateway with MCP protocol and Cognito JWT authentication
- Added a Web Search connector target — no schema or endpoint configuration needed
- Verified the setup with raw `tools/list` and `tools/call` MCP invocations

**Next**: Run notebook `02-web-search-strands-agent.ipynb` to use this Gateway with a Strands AI agent.